# FloatDeck V4 分层生成端（实验候选，Colab T4）
**不是 Android 渲染更新。没有运行模型/人工检查前，不能宣称分层可用。**
输入 ZIP V1，使用原始图片与深度。人物和珍珠颜色仅来自原图；AI 仅补经审核遮罩内的背景。自动分割并不可靠，必须检查发丝、手指、透明纱和珍珠。
原始画面被裁切的珍珠、珍珠遮住的人物纹理无法无损恢复；本流程不会生成新人物或新珍珠伪装解决。
运行顺序：设置 T4 → 独立 Python 3.12 → 导入 → 分割/编辑 → 抠图/编辑 → 审核遮罩 → 背景补全 → 合成检查 → 审核/打包。

In [ ]:
from pathlib import Path
import os,sys,subprocess,json,shutil
BASE=Path('/content/floatdeck-v4');BASE.mkdir(exist_ok=True)
ROOT=Path('/content/FloatDeck-v4-generator')
# 只克隆生成端分支；不修改已安装的 FloatDeck APP。
if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','--branch','spatial-layered-v4-generator','--single-branch',
                    'https://github.com/fdsfbgdf-blip/FloatDeck.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin','spatial-layered-v4-generator'],check=True)
print('generator commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
# 若此前 DA3 有独立 Python 3.12，可填它作为 base；不会向该 DA3 环境安装/降级依赖。
DA3_PYTHON=''  # 例 '/content/da3-venv/bin/python'
base=DA3_PYTHON or sys.executable
version=subprocess.check_output([base,'-c','import sys; print("%d.%d"%sys.version_info[:2])'],text=True).strip()
if version!='3.12':
    subprocess.run([sys.executable,'-m','pip','install','-q','uv==0.6.3'],check=True)
    subprocess.run([sys.executable,'-m','uv','python','install','3.12'],check=True)
    base=subprocess.check_output([sys.executable,'-m','uv','python','find','3.12'],text=True).strip()
ENV=BASE/'python312';PY=str(ENV/'bin/python')
if not Path(PY).exists():subprocess.run([base,'-m','venv',str(ENV)],check=True)
assert subprocess.check_output([PY,'-c','import sys; print(sys.version_info[:2])'],text=True).strip()=='(3, 12)'
MODEL_ENV=os.environ.copy();MODEL_ENV.update(HF_HOME=str(BASE/'cache'),HF_HUB_DISABLE_TELEMETRY='1',TOKENIZERS_PARALLELISM='false')
WORK=BASE/'project-001';WORK.mkdir(exist_ok=True)
def run(command,*extra):
    subprocess.run([PY,'-m','tools.layered_v4.cli',command,'--work',str(WORK),*map(str,extra)],cwd=ROOT,env=MODEL_ENV,check=True)
print('所有模型命令只使用:',PY)
cache=Path(MODEL_ENV['HF_HOME']);cache.mkdir(parents=True,exist_ok=True)
probe=cache/'.write-check';probe.write_text('ok');probe.unlink()
if shutil.disk_usage(cache).free<25*2**30:raise RuntimeError('安装前至少保留25 GiB磁盘用于环境/模型/原生输出')
# nvidia-smi 缺失不会直接报 FileNotFoundError；CUDA 最终由独立解释器内 torch 验证。
smi=shutil.which('nvidia-smi')
if smi:
    r=subprocess.run([smi],capture_output=True,text=True,timeout=20);print(r.stdout if r.returncode==0 else 'nvidia-smi 不可用，稍后验证 torch CUDA')
else:print('nvidia-smi 未找到；不能据此断言 CUDA 成功或失败。请确认 Colab 已选择 T4。')


## 安装与预检查
首次下载需要网络、约 20 GiB 以上可用磁盘。Torch 固定为支持 Python 3.12 的 CUDA 12.4 轮子；显式安装 `addict`。不向 Notebook 内核或既有 DA3 环境安装模型依赖。
`preflight` 不通过时停止，不继续生成“假背景”。

In [ ]:
subprocess.run([PY,'-m','pip','install','--upgrade','pip'],check=True)
subprocess.run([PY,'-m','pip','install','torch==2.6.0','torchvision==0.21.0','--index-url','https://download.pytorch.org/whl/cu124'],check=True)
subprocess.run([PY,'-m','pip','install','-r',str(ROOT/'tools/layered_v4/requirements-colab.txt')],check=True)
subprocess.run([PY,'-m','pip','check'],check=True)
run('preflight')
run('lock-models')  # 将公开模型 main 解析为确切 commit，之后每阶段按该 revision 加载


## 上传实际 ZIP
上传 `wallpaper_001.spatial.zip`。源图/深度保持同尺寸、同方向；遇到不一致或 EXIF 旋转会明确停止，不擅自旋转一张图。项目重跑请更换 WORK，防止覆盖已审核文件。

In [ ]:
from google.colab import files
uploaded=files.upload()
if len(uploaded)!=1:raise ValueError('只上传一个输入 ZIP')
INPUT=BASE/'input.spatial.zip';INPUT.write_bytes(next(iter(uploaded.values())))
run('init','--input',INPUT)
from PIL import Image
from IPython.display import display
source=Image.open(WORK/'source.png');print('原始像素尺寸:',source.size)
preview=source.copy();preview.thumbnail((560,1000));display(preview)
print((WORK/'input.json').read_text())


## 分割建议（必须编辑审核）
配置中的框和局部区域基于本轮仙女图。更换图片必须改框。SAM 输出三候选和分数，分数不等于正确；需把飞散发丝、光球旁手指和裙摆纳入人物，珍珠只保留球体，不把岩石或白纱当珍珠。
每次 `run` 是单独进程；SAM 结束后退出，CUDA 内存不与 VitMatte/SDXL 共存。

In [ ]:
# 分割修复：旧项目 config 不会自动升级，显式迁移场景提示后再运行。
# 先确认第一单元已拉取最新分支；无需重新上传/初始化已有 source。
import copy
from PIL import Image
from IPython.display import display
from google.colab import files
source=Image.open(WORK/'source.png')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from tools.layered_v4.segmentation import PROMPTS,previews
cfg=json.loads((WORK/'config.json').read_text())
RESET_SCENE_PROMPTS=True # 首次修复旧项目为 True；再次调整后设 False 保留手动提示
if RESET_SCENE_PROMPTS:cfg['sam_prompts']=copy.deepcopy(PROMPTS)
# 可改每个对象的 box / positive / negative，全部是原图归一化 XY。
cfg['boxes']={k:v['box'] for k,v in cfg['sam_prompts'].items()}
(WORK/'config.json').write_text(json.dumps(cfg,ensure_ascii=False,indent=2))
def show_masks(names):
    for name in names:
        path=WORK/name
        if path.exists():
            print('文件 / 图层:',name);im=Image.open(path).copy();im.thumbnail((650,1050));display(im)
previews(WORK,cfg)
show_masks(['source.png','segmentation-previews/person_prompts.png','segmentation-previews/pearl_prompts.png'])
run('segment') # 仅生成候选，不自动选择，不执行任何抠图/补全
for name in ['person','pearl']:
    print('目标:',name, (WORK/f'{name}_proposal.json').read_text())
    for i in range(3):show_masks([f'{name}_candidate_{i}.png',f'segmentation-previews/{name}_candidate_{i}_overlay.png'])


In [ ]:
# 逐个查看所有带名称候选；填入索引 0/1/2。无合格候选则调整提示重跑上一单元。
SELECT={'person':None,'pearl':None}
for name,index in SELECT.items():
    if type(index) is not int or index not in range(3):raise RuntimeError('必须人工选择 '+name+' 候选，禁止自动按分数选择')
for name,index in SELECT.items():shutil.copyfile(WORK/f'{name}_candidate_{index}.png',WORK/f'{name}_mask.png')
previews(WORK,cfg)
show_masks(['source.png','person_mask.png','segmentation-previews/person_mask_overlay.png','pearl_mask.png','segmentation-previews/pearl_mask_overlay.png'])
print('尚未人工审核通过；可在下面画笔修正原生尺寸遮罩。')


### 原生坐标编辑器 / 外部软件
白=前景，黑=背景；Trimap 灰=未知，Alpha 灰=半透明。可反复更换 EDIT_FILE 并运行编辑器，缩放后逐笔修正。保存不会自动批准。
也可下载 PNG 在 GIMP/Krita 编辑，然后用下一格上传。必须保持原生宽高，L 灰度 PNG；不要上传缩略图、黑底 RGB 抠图或截图。

In [ ]:
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from tools.layered_v4.editor import display_editor
EDIT_FILE='person_mask.png'  # pearl_mask.png / 后续的 person_trimap.png、person_alpha.png 等
# 显式使用 Colab Notebook 内核只显示/编辑图片，不在内核加载 GPU 模型。
display_editor(WORK,EDIT_FILE)


In [ ]:
# 可选：替换一个或多个经外部编辑的遮罩；不执行时保留编辑器已保存文件。
UPLOAD_EDITED_MASKS=False
if UPLOAD_EDITED_MASKS:
    from tools.layered_v4.editor import ALLOWED
    edited=files.upload()
    for name,data in edited.items():
        if name not in ALLOWED:raise ValueError('文件名不在遮罩白名单: '+name)
        import io
        im=Image.open(io.BytesIO(data))
        if im.size!=source.size or im.mode!='L':raise ValueError('必须保持原生尺寸、8-bit L 通道')
        (WORK/name).write_bytes(data)


## 精细抠图 / 白纱 Alpha
先确认分割，再运行 VitMatte。默认 768 tile / 128 overlap，原生坐标、不缩放原图。配置中的 translucent_regions 是本图白纱不确定区提示，会把支持内该区域设为灰 trimap，绝非透明度真值；更换照片必须修改。三值 trimap 可编辑；脸、眼睛、手指核心及珍珠反光内部应为确定前景。
VitMatte 不是透明白纱的真值。薄纱处需调 `person_alpha.png`，不要把半透明全部涂成白色；保护核心 `person_protect.png` 强制保留原图 RGB。球体内部保持不透明，不把高光当透明。
若修改了已有 trimap，再运行 matte，并同步检查 person_protect.png：旧保护区仍会强制不透明，必要时擦除。若直接修改 alpha，不要再跑 matte 覆盖。

In [ ]:
# 本次仅修复 SAM。保持暂停，禁止继续 VitMatte/补全/打包。
CONTINUE_AFTER_SEGMENTATION=False
run('check-segmentation')
SEGMENTATION_REVIEWED=False # 查看命名遮罩、叠加图及所有细节后才可手动确认
if not SEGMENTATION_REVIEWED:raise RuntimeError('请先人工检查分割；本次停在此处')
run('approve-segmentation','--reviewed')
if not CONTINUE_AFTER_SEGMENTATION:raise RuntimeError('本次 SAM 修复暂停后续流程')
run('matte')
run('prepare-masks')
show_masks(['person_alpha.png','pearl_alpha.png','person_protect.png','pearl_protect.png','background_hole.png','person_unobserved.png'])


## 批准当前遮罩，再补全背景
检查所有 Alpha、保护核心、背景 hole 和 `person_unobserved.png`。如果 alpha 修改扩大了支持范围，请同步编辑 hole；或显式删除旧 hole 后重跑 prepare-masks 重新从 alpha 生成。
Hole 内允许生成背景；其外最后逐像素复制原始可见背景，不能被模型改写。不要只为掩盖接缝随意扩大 hole。
人物与珍珠互相遮挡的未知人物区域不会填入珍珠纹理。源图下边缘裁切的珍珠不能凭空补完整球体。
遮罩审核后任何修改都会使哈希失效，必须重新检查批准。

In [ ]:
if not globals().get('CONTINUE_AFTER_SEGMENTATION',False):raise RuntimeError('本次 SAM 修复禁止运行后续流程')
MASKS_REVIEWED=False # 编辑并检查后手动设 True
if not MASKS_REVIEWED:raise RuntimeError('遮罩/透明白纱/珍珠轮廓尚未审核')
run('approve-masks','--reviewed')
run('background') # SDXL: global + native tiled refine；一次只驻留这一模型，T4 使用 CPU offload
show_masks(['background.png'])
print((WORK/'inpaint-execution.json').read_text())


## 合成检查：零位移、分别位移、局部放大、黑白底
人物和珍珠从原始 RGB 取色；仅对部分 Alpha 像素进行基于补全背景的 straight-alpha 去污染。该逆问题不唯一，色域裁切/重合误差会记录，不能宣称“数学重合即真实抠图”。
必须检查 background 中没有残留人物、珍珠、裙摆/发丝幽灵；分别位移揭露补全区域。未知人物区域另有棋盘诊断预览，不伪装成已恢复的衣料。

In [ ]:
if not globals().get('CONTINUE_AFTER_SEGMENTATION',False):raise RuntimeError('本次 SAM 修复禁止运行后续流程')
run('build')
print((WORK/'quality.json').read_text())
for path in sorted((WORK/'previews').glob('*.png')):
    print(path.name);im=Image.open(path).copy();im.thumbnail((1200,900));display(im)


## 最终人工验收 / 打包
每项都需检查原始尺寸局部：脸/眼睛、手指数量姿态、细发丝、白纱、裙摆、珍珠球形和反光、背景无幽灵、位移露底、黑白底 Alpha。
未通过就编辑遮罩/trimap/Alpha/保护核心，重新批准遮罩、补全和合成；不要勾选虚假的通过。
默认仅导出 UNREVIEWED-DRAFT。已审核实验包也不是 Android ZIP V1，不能直接导入现有 APP。

In [ ]:
if not globals().get('CONTINUE_AFTER_SEGMENTATION',False):raise RuntimeError('本次 SAM 修复禁止运行后续流程')
CHECKS={k:False for k in ['segmentation','hair','face_fingers','veil_alpha','skirt','pearl','background_no_ghosts','shift_previews','alpha_on_black_white']}
ACCEPT_UNKNOWN_PERSON_BEHIND_PEARL=False # 仅确实接受已知缺口时，才显式确认；不会生成隐藏人物
FINAL_REVIEWED=all(CHECKS.values())
OUT=BASE/('wallpaper_001.spatial-v2.zip' if FINAL_REVIEWED else 'wallpaper_001.UNREVIEWED.spatial-v2.zip')
if FINAL_REVIEWED:
    flags=['--reviewed']+(['--allow-known-person-occlusion'] if ACCEPT_UNKNOWN_PERSON_BEHIND_PEARL else [])
    run('approve-output',*flags)
run('pack','--output',OUT,*([] if FINAL_REVIEWED else ['--draft']))
subprocess.run([PY,'-m','tools.layered_v4.validate',str(OUT),*([] if FINAL_REVIEWED else ['--allow-draft'])],cwd=ROOT,env=MODEL_ENV,check=True)
files.download(str(OUT))


## 故障与边界
- 找不到 nvidia-smi：以独立解释器 torch.cuda 探测为准，不执行不存在的命令。
- No module named addict：确认 pip 安装与模型命令都用上方 PY，而非 Notebook 内核；重新运行 requirements 安装和 preflight。
- CUDA OOM：停止其他模型/旧进程；保持一次一个 stage；把 matte_tile 降为 512、inpaint_tile 降为 768（overlap 仍小于 tile）、global_long_side 改 896 后重跑；不要降低最终 PNG 尺寸或以模糊/复制替代补全。性能和模型质量仍需实际 T4 执行记录。
- 模型下载失败/访问受限：停止，检查缓存、网络和 Hugging Face 权限；禁止用拷贝原图/传统模糊假装 AI 背景。
- 透明衣料和被珍珠遮住的人物纹理无法仅凭单图唯一求解，完整人物恢复需要额外资料；不能重新生成人物替代。
详细格式与限制见仓库 `SPATIAL_V4_GENERATOR_REPORT.md`、`tools/layered_v4/FORMAT_V2_DRAFT.md`。